*Title:* 6_Upset
*Goal:* Make upset plots for the spatially-repeated outliers in 2016 and 2017 (2 separate plots)

# 6.1_Upset_Data 
*Goal:* Generate datasets 
based on pub_upset.

``` R
##### 1a Generate datasets (spatial 2016 0r 2017):
library(data.table)
library(tidyverse)

file<-fread("repeatability_cleaned_AGAIN.tsv",sep="\t",header=T)

data_2016<-file %>%
select("ColumnNumber","only_2016_spatial")%>%
filter(only_2016_spatial>1) #I'm only interested in the sites that are spatially repeated (ie 2+)

data_2017<-file %>%
select("ColumnNumber","only_2017_spatial")%>%
filter(only_2017_spatial>1)

#Save data to a TSV file
fwrite(data_2017, file = "only_2017_spatial_repeated.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote=F)
fwrite(data_2016, file = "only_2016_spatial_repeated.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote=F)
```
-------------------------------------
*Goal:* Clean the datasets. Filter to only contain SNPs that are spatial (i.e. in the only_2016_spatial_repeated.tsv file)

```bash
# 2016 filter:
awk '
FILENAME == "repeatability/only_2016_spatial_repeated.tsv" {
#Store each value from all rows of file1 in an associative array
a[$1]
next
}
FILENAME == "4.outliers_poolfstat_SNP_FST_with_locations_95_2016.tsv" {
#Process the header row to determine which columns to print
if (FNR == 1) {
header = $1 "\t" # Always include the first column (population)
for (i = 2; i <= NF; i++) {
if ($i in a) {
cols_to_print[i] = 1
header = header $i "\t"
}
}
print header
} else {
#Print the entire row for the matching columns
row = $1 "\t" # Always include the first column (population)
for (i = 2; i <= NF; i++) {
if (i in cols_to_print) {
row = row $i "\t"
}
}
print row
}
}
' only_2016_spatial_repeated.tsv 4.outliers_poolfstat_SNP_FST_with_locations_95_2016.tsv > 4.outliers_poolfstat_SNP_FST_with_locations_95_2016_outliers_spatial_pure.tsv
```
``` R
#Looks like output file has a random empty colum nat end. Need to remove it.
file<- 4.outliers_poolfstat_SNP_FST_with_locations_95_2016_outliers_spatial_pure.tsv
file2<-file[,c(1:161797)]
fwrite(file2, file = "4.outliers_poolfstat_SNP_FST_with_locations_95_2016_outliers_spatial_pure_cleaned.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote = FALSE)
``` 
``` bash
# Filter 2017:
awk '
FILENAME == "only_2017_spatial_repeated.tsv" {
#Store each value from all rows of file1 in an associative array
a[$1]
next
}
FILENAME == "4.outliers_poolfstat_SNP_FST_with_locations_95_2017.tsv" {
#Process the header row to determine which columns to print
if (FNR == 1) {
header = $1 "\t" # Always include the first column (population)
for (i = 2; i <= NF; i++) {
if ($i in a) {
cols_to_print[i] = 1
header = header $i "\t"
}
}
print header
} else {
#Print the entire row for the matching columns
row = $1 "\t" # Always include the first column (population)
for (i = 2; i <= NF; i++) {
if (i in cols_to_print) {
row = row $i "\t"
}
}
print row
}
}
' only_2017_spatial_repeated.tsv 4.outliers_poolfstat_SNP_FST_with_locations_95_2017.tsv > 4.outliers_poolfstat_SNP_FST_with_locations_95_2017_outliers_spatial_pure.tsv
```

# 6.2_Upset_Plot
*Goal:* Generate upset plots

```R
# 2016 plots:
#Data:
library("ComplexUpset")
library("data.table")
library("ggplot2")
library("tidyverse")

#Need to load and transpose the data for the package:
data <- fread("4.outliers_poolfstat_SNP_FST_with_locations_95_2016_outliers_spatial_pure_cleaned.tsv", header = TRUE, sep = "\t") %>% as.data.frame() 
rownames(data) <- data[, 1] #make populations the rownames
data <- data[, -1] #removing population column
data<- t(data) #transpose dataframe
if (!is.data.frame(data)) {
  data <- as.data.frame(data)
} #for some reason need to force dataframe again
str(data) #should be TRUE!!

#Make sure data has no NA:
any(is.na(data))
data[is.na(data), ]
nrow(data)

data<-na.omit(data) #removec NA 
any(is.na(data))
data[is.na(data), ]
nrow(data)
# Graph will show intersection across estuaries of outliers (min intersection of 2)
plot<- upset(data, colnames(data), name='Estuaries', width_ratio=0.05, min_size=1, min_degree= 2, height_ratio=0.5,
base_annotations=list('Intersection size'=intersection_size(
            text=list(
                size= 2
            )
        )
        + annotate(
            geom='text', x=Inf, y=Inf,
            label=paste('Total:', nrow(data)),
            vjust=1, hjust=1
        )
        + ylab('Intersection size')
    ),  
    sort_sets=FALSE)  #this keeps the sets in alph. order (rather than by size)
ggsave("upset_2016_outliers_only.png", plot, width = 10, height =6, dpi = 500)
ggsave("upset_2016_outliers_only.pdf", plot, width = 10, height =6, dpi = 500)
ggsave("upset_2016_outliers_wide_only.png", plot, width = 40, height =40, dpi = 500)
ggsave("upset_2016_outliers_wide_only.pdf", plot, width = 40, height =40, dpi = 500)


print("graphs done")

# Data:
updata<- upset_data(data, colnames(data), min_size=1, min_degree= 2)

exclusive_intersection_df <- data.frame(
  Interaction = names(updata$sizes$exclusive_intersection),
  Count = updata$sizes$exclusive_intersection
) #just saving the exclusinve interactions counts (with the interactions names)


# Save the exclusive intersection sizes as a TSV file
write.table(exclusive_intersection_df, file = "upset_2016_outliers_exclusive_intersection_sizes_only.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote= F)

print("interactions data done")

### 2017 plots:
library("ComplexUpset")
library("data.table")
library("ggplot2")
library("tidyverse")

#Need to load and transpose the data for the package:
data <- fread("4.outliers_poolfstat_SNP_FST_with_locations_95_2017_outliers_spatial_pure_clean.tsv", header = TRUE, sep = "\t") %>% as.data.frame() 
rownames(data) <- data[, 1] #make populations the rownames
data <- data[, -1] #removing population column
data<- t(data) #transpose dataframe
if (!is.data.frame(data)) {
  data <- as.data.frame(data)
} #for some reason need to force dataframe again
str(data) #should be TRUE!!
nrow(data)
any(is.na(data))
data[is.na(data), ]

#I think there is a random row (fake SNP) at the end, where all are NA
data<-na.omit(data) #removec NA 
any(is.na(data))
data[is.na(data), ]
nrow(data)

# Graph will show intersection across estuaries of outliers (min intersection of 2)
plot<- upset(data, colnames(data), name='Estuaries', width_ratio=0.05, min_size=1, min_degree= 2, height_ratio=0.5,
base_annotations=list('Intersection size'=intersection_size(
            text=list(
                size= 2
            )
        )
        + annotate(
            geom='text', x=Inf, y=Inf,
            label=paste('Total:', nrow(data)),
            vjust=1, hjust=1
        )
        + ylab('Intersection size')
    ),  
    sort_sets=FALSE)  #this keeps the sets in alph. order (rather than by size)
ggsave("upset_2017_outliers_only.png", plot, width = 10, height =6, dpi = 500)
ggsave("upset_2017_outliers_only.pdf", plot, width = 10, height =6, dpi = 500)
ggsave("upset_2017_wide_outliers_only.png", plot, width = 40, height =40, dpi = 500)
ggsave("upset_2017_wide_outliers_only.pdf", plot, width = 40, height =40, dpi = 500)
print("graphs done")

# Data:
updata<- upset_data(data, colnames(data), min_size=1, min_degree= 2)

exclusive_intersection_df <- data.frame(
  Interaction = names(updata$sizes$exclusive_intersection),
  Count = updata$sizes$exclusive_intersection
) #just saving the exclusinve interactions counts (with the interactions names)


# Save the exclusive intersection sizes as a TSV file
write.table(exclusive_intersection_df, file = "upset_2017_outliers_exclusive_intersection_sizes_only.tsv", sep = "\t", row.names = FALSE, col.names = TRUE, quote= F)

print("interactions data done")
```

# 6.3_Upset_Top_Plot
*Goal:* Generate a top upset plot (allowing for colouring by type)

``` R
# Make a barplot in ggplot2 using the count column for 2016:
library(ggplot2)
library(tidyverse)
library(data.table)
library(patchwork)
library(cowplot)
library(patchwork)

file<-"interactions_exclusive_2016_only.tsv"
data <- fread(file, header = TRUE, sep = "\t") %>% as.data.frame() 
data <- data %>% mutate_all(~ str_replace_all(., "\\s+", "")) #spaces are randomly in the dataframe
data$Count<-as.numeric(data$Count)
data$Bar_ID <- as.numeric(data$Bar_ID)


# Create a barplot using ggplot2 (2016:)
main_plot <- ggplot(data, aes(x = Bar_ID, y = Count, fill = Colour)) +
  geom_bar(stat = "identity", width = 0.5) +
  theme_minimal() +
  scale_fill_identity(guide = "legend",labels = c("Spatial 6","Spatial 5", "Spatial 4","Spatial 3","Spatial 2")) + 
  labs(y = "Interaction Size (Count)", x = "2016 Interactions", fill = "Repeatability Type") +
  scale_y_continuous(limits = c(-100,10500), breaks = seq(0,10500, by = 500), expand = c(0, 0)) +  # Add y-axis ticks every 500 and remove space
  scale_x_continuous(expand = c(0.005, 0)) +  # Remove space on x-axis
  theme(panel.grid.major = element_blank(),  # Remove major grid lines
        panel.grid.minor = element_blank(),
        panel.background = element_blank(), 
        axis.line = element_line(colour = "black"),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),  # Remove x-axis ticks
        axis.ticks.y = element_line(color = "black"),
        legend.justification = c("right", "top"),
       legend.position =c(2,2)) + 
  annotate("rect", xmin = 50.5, xmax = 58, ymin = -100, ymax = 100, color = "black", linetype = "solid", linewidth = 0.25, fill = NA) + #adding rectangular box around 
  annotate("segment", x = 50.5, xend = 32.2, y = 100, yend = 4300, color = "black", linetype = "dashed") + #adding lines to eventually connect the inset 
  annotate("segment", x = 58, xend = 46.2, y = 100, yend = 4300, color = "black", linetype = "dashed") #adding lines to eventually connect the inset 

#patchwork verison of inset plot:
inset_data<-filter(data,Interaction_Number>4)
inset_barplot <- ggplot(inset_data, aes(x = Bar_ID, y = Count, fill = Colour)) +
  geom_bar(stat = "identity", width = 0.5) +
  scale_fill_identity() +  # Use the exact colors specified in the Colour column
  theme_minimal() +
  scale_y_continuous(limits = c(0, 20)) +  # Set the y-axis limits 
  theme(axis.text.x = element_blank(),
  panel.grid = element_blank(),  
        axis.ticks.x = element_blank(),  # Remove x-axis ticks
        axis.title.x = element_blank(),  # Remove x-axis title
        axis.title.y = element_blank(), # Remove y-axis title
        panel.border = element_rect(color = "black", fill = NA))   # Add a solid border


combined_plot <- main_plot +
  inset_element(inset_barplot, left = 0.5, bottom = 0.4, right = 0.8, top = 1) +
  plot_annotation(tag_levels = list(c('A', 'B')))

legend_plot_with_label <- ggdraw() +
  draw_grob(legend) +
  scale_fill_identity(
  guide = "legend",
  breaks = c("#0041B1","#A200B1","#B10026","#E31A1C","#ef6604"), # actual hex values used in Colour
  labels = c("Spatial 6","Spatial 5","Spatial 4","Spatial 3","Spatial 2"),
  name = "Repeatability Type"
) 
ggsave("NEW_upset_top_plot_2016_only.png", legend_plot_with_label , width = 10, height = 6, dpi = 700)

# Combine the main plot, inset plot, and legend
final_plot <- combined_plot + inset_element(legend_plot_with_label, left = 0.82, bottom = 0.75, right = 0.98, top = 0.88)

# Save the final plot
ggsave("upset_top_plot_2016_only.png", final_plot, width = 10, height = 6, dpi = 1000)
ggsave("upset_top_plot_2016_only.pdf", final_plot, width = 10, height = 6, dpi = 1000)

## 2017 manual plots:
#No legend, labelled "c" and "d":
library(ggplot2)
library(tidyverse)
library(data.table)
library(patchwork)

file<-"interactions_exclusive_2017_only.tsv"
data <- fread(file, header = TRUE, sep = "\t",quote=F) %>% as.data.frame() 
data$Count<-as.numeric(data$Count)
data$Bar_ID <- as.numeric(data$Bar_ID)

main_plot <- ggplot(data, aes(x = Bar_ID, y = Count, fill = Colour)) +
  geom_bar(stat = "identity", width = 0.5) +
  theme_minimal() +
  scale_fill_identity(guide = "none", labels = c("6 Estuaries","5 Estuaries", "4 Estuaries","3 Estuaries","2 Estuaries","1 Estuary")) + 
  labs(y = "Interaction Size (Count)", x = "2017 Interactions", fill = "Spatial Repeatability") +
  scale_y_continuous(limits = c(-100, 10500), breaks = seq(0,10500, by = 500), expand = c(0, 0)) +  # Add y-axis ticks every 500 and remove space
  scale_x_continuous(expand = c(0.005, 0)) +  # Remove space on x-axis
  theme(panel.grid.major = element_blank(),  # Remove major grid lines
        panel.grid.minor = element_blank(),
        panel.background = element_blank(), 
        axis.line = element_line(colour = "black"),
        axis.text.x = element_blank(),
        axis.ticks.x = element_blank(),  # Remove x-axis ticks
        axis.ticks.y = element_line(color = "black")) + 
  annotate("rect", xmin = 50.5, xmax = 58, ymin = -100, ymax = 100, color = "black", linetype = "solid", linewidth = 0.25, fill = NA) + #adding rectangular box around 
  annotate("segment", x = 50.5, xend = 32.5, y = 100, yend = 4300, color = "black", linetype = "dashed") + 
  annotate("segment", x = 58, xend = 46.5, y = 100, yend = 4300, color = "black", linetype = "dashed") 

#patchwork verison of inset plot:
inset_data<-filter(data,Interaction_Number>4)
inset_barplot <- ggplot(inset_data, aes(x = Bar_ID, y = Count, fill = Colour)) +
  geom_bar(stat = "identity", width = 0.5) +
  scale_fill_identity() +  # Use the exact colors specified in the Colour column
  theme_minimal() +
    scale_y_continuous(limits = c(0, 20))+
  theme(axis.text.x = element_blank(),
  panel.grid = element_blank(),  
        axis.ticks.x = element_blank(),  # Remove x-axis ticks
        axis.title.x = element_blank(),  # Remove x-axis title
        axis.title.y = element_blank(), # Remove y-axis title
        panel.border = element_rect(color = "black", fill = NA))   # Add a solid border


combined_plot <- main_plot +
  inset_element(inset_barplot, left = 0.5, bottom = 0.4, right = 0.8, top = 1) +
  plot_annotation(tag_levels = list(c('C', 'D')))

# Save the final plot
ggsave("upset_top_plot_2017_only.png", combined_plot, width = 10, height = 6, dpi = 1000)
```